In [1]:
import os
import cv2
import re
import timm
import torch
import numpy as np
import pandas as pd

from pathlib import Path
from tqdm.auto import tqdm

import albumentations as A
from albumentations.pytorch import ToTensorV2

from torch.utils.data import Dataset, DataLoader


d:\BDC_2026_PP\P_PP\poetri\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
CFG = {

    "batch_size":32,

    "num_workers":0,

    "num_classes":3,

    "device":"cuda"

}

DATA_DIR = Path("../data")

TRAIN_CSV = Path("train_folds.csv")

TEST_DIR = DATA_DIR / "test"

WEIGHT_DIR = Path("../weights")

OUTPUT_DIR = Path("../oof")

OUTPUT_DIR.mkdir(
    exist_ok=True,
    parents=True
)

In [3]:
MODELS = {

    "ConvNeXt":{

        "model_name":"convnextv2_tiny.fcmae_ft_in22k_in1k",

        "img_size":384,

        "weight_pattern":"best_fold{fold}.pth"

    },

    "Swin":{

        "model_name":"swinv2_tiny_window16_256",

        "img_size":256,

        "weight_pattern":"swin_best_fold{fold}.pth"

    },

    "SigLIP":{

        "model_name":"vit_base_patch16_siglip_384",

        "img_size":384,

        "weight_pattern":"siglip_best_fold{fold}.pth"

    }

}

In [4]:
df = pd.read_csv(TRAIN_CSV)

print(df.shape)

display(df.head())


(24853, 4)


,filepath,label,class_name,fold
0,..\data\train\0_Recyclable\R_1.jpg,0,0_Recyclable,2
1,..\data\train\0_Recyclable\R_10.jpg,0,0_Recyclable,2
2,..\data\train\0_Recyclable\R_100.jpg,0,0_Recyclable,2
3,..\data\train\0_Recyclable\R_1000.jpg,0,0_Recyclable,3
4,..\data\train\0_Recyclable\R_1001.jpg,0,0_Recyclable,3


In [5]:
def natural_sort_key(path):

    return [

        int(x) if x.isdigit() else x.lower()

        for x in re.split(r"(\d+)", path.name)

    ]


test_files = list(TEST_DIR.glob("*"))

test_files.sort(
    key=natural_sort_key
)

print(len(test_files))


1458


In [6]:
class ImageDataset(Dataset):

    def __init__(

        self,

        files,

        labels=None,

        transform=None

    ):

        self.files = files

        self.labels = labels

        self.transform = transform

    def __len__(self):

        return len(self.files)

    def __getitem__(self, idx):

        image = cv2.imread(str(self.files[idx]))

        image = cv2.cvtColor(

            image,

            cv2.COLOR_BGR2RGB

        )

        if self.transform is not None:

            image = self.transform(

                image=image

            )["image"]

        if self.labels is None:

            return image

        return image, self.labels[idx]

In [7]:
def build_transform(img_size):

    return A.Compose([

        A.LongestMaxSize(

            max_size=img_size

        ),

        A.PadIfNeeded(

            min_height=img_size,

            min_width=img_size,

            border_mode=cv2.BORDER_CONSTANT

        ),

        A.Normalize(),

        ToTensorV2()

    ])

In [8]:
def build_model(model_name):

    model = timm.create_model(
        model_name,
        pretrained=False,
        num_classes=CFG["num_classes"]
    )

    return model.to(CFG["device"])

In [9]:
def build_loader(files, labels=None, img_size=384):

    transform = build_transform(img_size)

    dataset = ImageDataset(
        files=files,
        labels=labels,
        transform=transform
    )

    loader = DataLoader(
        dataset,
        batch_size=CFG["batch_size"],
        shuffle=False,
        num_workers=CFG["num_workers"],
        pin_memory=True
    )

    return loader

In [10]:
def predict_loader(model, loader):

    model.eval()

    predictions = []

    with torch.no_grad():

        for batch in tqdm(loader):

            if isinstance(batch, (list, tuple)):
                images = batch[0]
            else:
                images = batch

            images = images.to(CFG["device"])

            logits = model(images)

            predictions.append(
                logits.cpu().numpy()
            )

    predictions = np.concatenate(predictions)

    return predictions

In [11]:
def load_fold_model(model_cfg, fold):

    model = build_model(
        model_cfg["model_name"]
    )

    weight_path = (
        WEIGHT_DIR /
        model_cfg["weight_pattern"].format(fold=fold)
    )

    print(weight_path)

    model.load_state_dict(
        torch.load(
            weight_path,
            map_location=CFG["device"]
        )
    )

    return model

In [12]:
def predict_valid_fold(model_cfg, valid_df, fold):

    print("=" * 60)
    print(f"Validation Fold {fold}")

    model = load_fold_model(model_cfg, fold)

    files = [
        Path(p)
        for p in valid_df["filepath"].values
    ]

    labels = valid_df["label"].values

    loader = build_loader(
        files,
        labels,
        img_size=model_cfg["img_size"]
    )

    logits = predict_loader(
        model,
        loader
    )

    return logits

In [13]:
def predict_test_fold(model_cfg, fold):

    print("=" * 60)
    print(f"Test Fold {fold}")

    model = load_fold_model(model_cfg, fold)

    loader = build_loader(
        test_files,
        img_size=model_cfg["img_size"]
    )

    logits = predict_loader(
        model,
        loader
    )

    return logits

In [14]:
def generate_oof(model_key):

    model_cfg = MODELS[model_key]

    print("=" * 80)
    print(f"Generating OOF : {model_key}")
    print("=" * 80)

    # OOF probability
    oof = np.zeros(
        (len(df), CFG["num_classes"]),
        dtype=np.float32
    )

    # Ground truth
    y_true = df["label"].values.astype(np.int64)

    for fold in sorted(df["fold"].unique()):

        valid_df = (
            df[df["fold"] == fold]
            .copy()
            .reset_index()
        )

        logits = predict_valid_fold(
            model_cfg,
            valid_df,
            fold
        )

        # simpan ke index asli
        oof[valid_df["index"].values] = logits

    np.save(
        OUTPUT_DIR / f"{model_key.lower()}_oof_logits.npy",
        oof
    )

    np.save(
        OUTPUT_DIR / "y_true.npy",
        y_true
    )

    print("Saved:")
    print(f"{model_key.lower()}_oof_logits.npy")
    print("y_true.npy")

In [15]:
def generate_test(model_key):

    model_cfg = MODELS[model_key]

    print("=" * 80)
    print(f"Generating TEST : {model_key}")
    print("=" * 80)

    test_logits = np.zeros(
        (len(test_files), CFG["num_classes"]),
        dtype=np.float32
    )

    folds = sorted(df["fold"].unique())

    for fold in folds:

        logits = predict_test_fold(
            model_cfg,
            fold
        )

        test_logits += logits

    test_logits /= len(folds)

    np.save(
        OUTPUT_DIR / f"{model_key.lower()}_test_logits.npy",
        test_logits
    )

    print(f"Saved : {model_key.lower()}_test_logits.npy")


    return test_logits

In [16]:
for model_name in MODELS:

    print("="*80)
    print(model_name)
    print("="*80)

    generate_oof(model_name)

    generate_test(model_name)

print("ALL DONE!")

ConvNeXt
Generating OOF : ConvNeXt
Validation Fold 0
..\weights\best_fold0.pth


100%|██████████| 156/156 [02:04<00:00,  1.25it/s]


Validation Fold 1
..\weights\best_fold1.pth


100%|██████████| 156/156 [01:59<00:00,  1.30it/s]


Validation Fold 2
..\weights\best_fold2.pth


100%|██████████| 156/156 [02:06<00:00,  1.23it/s]


... (51 outputs left out)

100%|██████████| 46/46 [01:26<00:00,  1.88s/it]


Test Fold 4
..\weights\siglip_best_fold4.pth


100%|██████████| 46/46 [01:27<00:00,  1.91s/it]

Saved : siglip_test_logits.npy
ALL DONE!
